In [1]:
%matplotlib qt
import mne

#mne.viz.set_3d_backend('pyvistaqt')
from mne.coreg import Coregistration
from mne.io import read_info


import numpy as np
#%matplotlib qt
import matplotlib
#matplotlib.use('qt5agg')  # Or any other backend you want to use

import matplotlib.pyplot as plt

import pandas as pd 
import os
from os.path import join as pathjoin
from pathlib import Path

import sys

from mne.preprocessing import ICA, corrmap, create_ecg_epochs, create_eog_epochs

from time import time

from autoreject import AutoReject

import glob

import psutil
import gc
from time import time

mne.set_log_level('INFO')

import json

from mne.channels import read_dig_polhemus_isotrak  # Función para leer archivos .pos

import re
# from mne.minimum_norm import apply_inverse, make_inverse_operator
#este codigo lo dejo comentado para acostumbrarme a su suso

import brainiak
from brainiak.isc import isc,bootstrap_isc

import statsmodels
from statsmodels.stats.multitest import multipletests

import pickle

from joblib import Parallel, delayed

In [2]:
try:
    # Si se ejecuta como SCRIPT .py: usar __file__
    sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))
except NameError:
    # Si se ejecuta como NOTEBOOK Jupyter: usar path relativo
    sys.path.append("..")  # sube un nivel desde la carpeta actual del notebook


# --- Configuración dinámica de rutas ---
from get_paths_MOUS import get_paths_MOUS

# Parámetros editables
disco = "g"
modality = "visual"
layer_script = "event"
subj = "sub-V1001"
#subj = sys.argv[1] ## name of participant list


# Generar variables automáticamente
path_dict = get_paths_MOUS(disco=disco, modality=modality, layer_script=layer_script, subj=subj)
globals().update(path_dict)

# Mostrar todos los paths generados
print("\n📁 Rutas generadas:")
for k, v in path_dict.items():
    print(f"{k:<20} → {v}")


📁 Rutas generadas:
datadir              → g:\MOUS_204\MOUS_visual
general_datadir      → g:\MOUS_204
output_preproc       → g:\MOUS_204\MOUS_visual\output_preproc
mri_dir              → g:\MOUS_204\sub-V1001\anat
meg_dir              → g:\MOUS_204\sub-V1001\meg
preproc_path         → g:\MOUS_204\MOUS_visual\output_preproc\preproc_event
channels_structure_path → g:\MOUS_204\channels_structure
epochs_path          → g:\MOUS_204\MOUS_visual\output_preproc\preproc_event\epochs_event
ICA_path             → g:\MOUS_204\MOUS_visual\output_preproc\preproc_event\ICA_event
epochs_clean_path    → g:\MOUS_204\MOUS_visual\output_preproc\preproc_event\epochs_clean_event
evoked_path          → g:\MOUS_204\MOUS_visual\output_preproc\preproc_event\evoked_event
source_path          → g:\MOUS_204\MOUS_visual\output_source\source_event
raw_hsp_path         → g:\MOUS_204\MOUS_visual\output_source\source_event\raw_hsp
fwd_path             → g:\MOUS_204\MOUS_visual\output_source\source_event\fwd
inverse_pat

In [3]:
#subjects = subj[:9]

subj = []

# Recorremos cada subdirectorio en la carpeta base
for subdirectorio in general_datadir.iterdir():
    # Comprobamos que el elemento sea un directorio y que su nombre comience con 'sub-A2'
    if modality == "visual":
        subject_prefix = 'sub-V1'
    elif modality == "auditory":
        subject_prefix = 'sub-A2'
        
    if subdirectorio.is_dir() and subdirectorio.name.startswith(subject_prefix):
        # Añadimos el nombre del sujeto a la lista
        subj.append(subdirectorio.name)

print(subj)
subjects= subj
subjects

['sub-V1001', 'sub-V1002', 'sub-V1003', 'sub-V1004', 'sub-V1005', 'sub-V1006', 'sub-V1007', 'sub-V1008', 'sub-V1009', 'sub-V1010', 'sub-V1011', 'sub-V1012', 'sub-V1013', 'sub-V1015', 'sub-V1016', 'sub-V1017', 'sub-V1019', 'sub-V1020', 'sub-V1022', 'sub-V1024', 'sub-V1025', 'sub-V1026', 'sub-V1027', 'sub-V1028', 'sub-V1029', 'sub-V1030', 'sub-V1031', 'sub-V1032', 'sub-V1033', 'sub-V1034', 'sub-V1035', 'sub-V1036', 'sub-V1037', 'sub-V1038', 'sub-V1039', 'sub-V1040', 'sub-V1042', 'sub-V1044', 'sub-V1045', 'sub-V1046', 'sub-V1048', 'sub-V1049', 'sub-V1050', 'sub-V1052', 'sub-V1053', 'sub-V1054', 'sub-V1055', 'sub-V1057', 'sub-V1058', 'sub-V1059', 'sub-V1061', 'sub-V1062', 'sub-V1063', 'sub-V1064', 'sub-V1065', 'sub-V1066', 'sub-V1068', 'sub-V1069', 'sub-V1070', 'sub-V1071', 'sub-V1072', 'sub-V1073', 'sub-V1074', 'sub-V1075', 'sub-V1076', 'sub-V1077', 'sub-V1078', 'sub-V1079', 'sub-V1080', 'sub-V1081', 'sub-V1083', 'sub-V1084', 'sub-V1085', 'sub-V1086', 'sub-V1087', 'sub-V1088', 'sub-V1089'

['sub-V1001',
 'sub-V1002',
 'sub-V1003',
 'sub-V1004',
 'sub-V1005',
 'sub-V1006',
 'sub-V1007',
 'sub-V1008',
 'sub-V1009',
 'sub-V1010',
 'sub-V1011',
 'sub-V1012',
 'sub-V1013',
 'sub-V1015',
 'sub-V1016',
 'sub-V1017',
 'sub-V1019',
 'sub-V1020',
 'sub-V1022',
 'sub-V1024',
 'sub-V1025',
 'sub-V1026',
 'sub-V1027',
 'sub-V1028',
 'sub-V1029',
 'sub-V1030',
 'sub-V1031',
 'sub-V1032',
 'sub-V1033',
 'sub-V1034',
 'sub-V1035',
 'sub-V1036',
 'sub-V1037',
 'sub-V1038',
 'sub-V1039',
 'sub-V1040',
 'sub-V1042',
 'sub-V1044',
 'sub-V1045',
 'sub-V1046',
 'sub-V1048',
 'sub-V1049',
 'sub-V1050',
 'sub-V1052',
 'sub-V1053',
 'sub-V1054',
 'sub-V1055',
 'sub-V1057',
 'sub-V1058',
 'sub-V1059',
 'sub-V1061',
 'sub-V1062',
 'sub-V1063',
 'sub-V1064',
 'sub-V1065',
 'sub-V1066',
 'sub-V1068',
 'sub-V1069',
 'sub-V1070',
 'sub-V1071',
 'sub-V1072',
 'sub-V1073',
 'sub-V1074',
 'sub-V1075',
 'sub-V1076',
 'sub-V1077',
 'sub-V1078',
 'sub-V1079',
 'sub-V1080',
 'sub-V1081',
 'sub-V1083',
 'sub-

In [4]:
def print5(*args):
    print(*(f"{x:.5f}" if isinstance(x, float) else x for x in args))

channels = pd.read_csv(channels_structure_path / f"channels_mag_{modality}.csv")
channels_mag=channels[channels[f"canal_efectivo_{modality}"].notna()][f"canal_efectivo_{modality}"]

channels_mag=channels_mag.tolist()
print5(channels_mag)
indice_channels_efectivos = channels[channels[f"canal_efectivo_{modality}"].notna()]["indice"]
indice_channels_efectivos=indice_channels_efectivos.tolist()
# del channels



# ------------------------------
# Parámetros sliding window y bootstrap
# ------------------------------
window_size_sec = 3.0     # segundos
sliding_window_sec = 0.351  # segundos
n_jobs = -1
threshold = 0.05

# summary_statistic_isc="median"

summary_statistic_bootstrap_isc="median"

['MLC11-4304', 'MLC12-4304', 'MLC13-4304', 'MLC14-4304', 'MLC15-4304', 'MLC16-4304', 'MLC17-4304', 'MLC21-4304', 'MLC22-4304', 'MLC23-4304', 'MLC24-4304', 'MLC25-4304', 'MLC31-4304', 'MLC32-4304', 'MLC41-4304', 'MLC42-4304', 'MLC51-4304', 'MLC52-4304', 'MLC53-4304', 'MLC54-4304', 'MLC55-4304', 'MLC61-4304', 'MLC62-4304', 'MLC63-4304', 'MLF11-4304', 'MLF12-4304', 'MLF13-4304', 'MLF14-4304', 'MLF21-4304', 'MLF22-4304', 'MLF23-4304', 'MLF24-4304', 'MLF25-4304', 'MLF31-4304', 'MLF32-4304', 'MLF33-4304', 'MLF34-4304', 'MLF35-4304', 'MLF41-4304', 'MLF42-4304', 'MLF43-4304', 'MLF44-4304', 'MLF45-4304', 'MLF46-4304', 'MLF51-4304', 'MLF52-4304', 'MLF53-4304', 'MLF54-4304', 'MLF55-4304', 'MLF56-4304', 'MLF61-4304', 'MLF63-4304', 'MLF64-4304', 'MLF65-4304', 'MLF66-4304', 'MLF67-4304', 'MLO11-4304', 'MLO12-4304', 'MLO13-4304', 'MLO14-4304', 'MLO21-4304', 'MLO22-4304', 'MLO23-4304', 'MLO24-4304', 'MLO31-4304', 'MLO32-4304', 'MLO33-4304', 'MLO34-4304', 'MLO41-4304', 'MLO42-4304', 'MLO43-4304', 'MLO4

In [5]:
from joblib import Parallel, delayed

def run_single_bootstrap_window_all(segment, seed,
                                    summary_statistic_isc=None,
                                    summary_statistic_bootstrap_isc='median',
                                    n_bootstraps=1000,
                                    ci_percentile=95,
                                    side='right'):
    
    iscs_window_all = isc(
        data=segment,
        pairwise=False,
        summary_statistic=summary_statistic_isc,
        tolerate_nans=True
    )
    
    iscs_bootstrap_window_all, ci_window_all, p_window_all, distribution_window_all = bootstrap_isc(
        iscs_window_all,
        pairwise=False,
        summary_statistic=summary_statistic_bootstrap_isc,
        n_bootstraps=n_bootstraps,
        ci_percentile=ci_percentile,
        side=side,
        random_state=seed
    )

    dict_isc_window_all = {
        "iscs_window_all": iscs_window_all,
        "iscs_bootstrap_window_all": iscs_bootstrap_window_all,
        "ci_window_all": ci_window_all,
        "p_window_all": p_window_all,
        "distribution_window_all": distribution_window_all
    }

    return dict_isc_window_all


In [6]:
# ------------------------------
# Loop principal por condición
# ------------------------------
n_subjects = len(subjects)


for cond in ["zinnen", "woorden"]:

    valid_arrays = []
    valid_subjects = []

    print(f"\n🔄 Procesando condición: {cond}")
    for i in range(n_subjects):
        subj = subjects[i]
        file_path = evoked_path / f"{subj}_evoked_{cond}_{layer_script}-ave.fif"

        try:
            evokeds = mne.read_evokeds(file_path)
            
            # Read evoked data
            evoked = evokeds[0]
            data_subj = evoked.pick("mag", exclude="bads").data  # (n_dipoles, n_times)
            data_subj_swapped = data_subj.T  # (n_times, n_dipoles)

            if np.all(data_subj_swapped == 0) or np.isnan(data_subj_swapped).all():
                print(f"{subj} only has 0 or Nants, it's omitted ")
                continue
            
            # append the data and subjects
            valid_arrays.append(data_subj_swapped)
            valid_subjects.append(subj)

        except FileNotFoundError:
            print(f"Archivo no encontrado para {subj}: {file_path}")
            continue
        except Exception as e:
            print(f"Error al procesar {subj}: {e}")
            continue
    
    # verify number of valid subjects
    if len(valid_arrays) < 2:
        print(f"No hay suficientes sujetos válidos para condición {cond}")
        continue
    
    #valid arrays contains a list of arrays with shape (n_times, n_channels)
    array = np.stack(valid_arrays, axis=2)  # (n_times, n_channels, n_subjects)
    
    print(f"Array ISC creado con forma {array.shape} ({len(valid_subjects)} sujetos válidos)")

    sfreq = evoked.info["sfreq"]
    n_samples, n_channels, _ = array.shape

    # we transform window size and sliding window to samples
    window_size_samples = int(window_size_sec * sfreq)
    sliding_window_all_samples = int(sliding_window_sec * sfreq)
    
    #begins in 0, moves in sliding steps, 
    # ends at n_times - window_size +1 to add the last point
    # Creamos segmentos y guardamos los intervalos en segundos
    segments = []
    window_intervals = []

    for start in range(0, n_samples - window_size_samples + 1, sliding_window_all_samples):
        segment = array[start:start + window_size_samples, :, :]
        if segment.shape[0] == window_size_samples:
            segments.append(segment)
            start_sec = start / sfreq
            end_sec = (start + window_size_samples) / sfreq
            window_intervals.append((start_sec, end_sec))
    
    window_number = len(segments)
    
    # Ejecutar en paralelo por cada segmento
    results = Parallel(n_jobs=n_jobs)(
        delayed(run_single_bootstrap_window_all)(segment, seed)
        for seed, segment in enumerate(segments)
    )

    # Convertir lista de dicts -> dict de listas
    dict_isc_window_all = {
        key: [r[key] for r in results]
        for key in results[0]
    }
    dict_isc_window_all["window_intervals"] = window_intervals
    
        
    #THESE LISTS WILL BE APPEND OND DICTIONARE
    significative_channels_all=[]
    num_significative_channels_all = []
    significative_channels_adjusted_all = []
    num_significative_channels_adjusted_all = []
    
    for i in range(window_number):
        iscs = dict_isc_window_all["iscs_window_all"][i]
        iscs_bootstrap = dict_isc_window_all["iscs_bootstrap_window_all"][i]
        ci = dict_isc_window_all["ci_window_all"][i]
        p = dict_isc_window_all["p_window_all"][i]
        distribution = dict_isc_window_all["distribution_window_all"][i]


        start_sample = i * sliding_window_all_samples
        start_sec = start_sample / sfreq
        end_sec = (start_sample + window_size_samples) / sfreq

        print(f"Window {i + 1}: {start_sec:.2f}s - {end_sec:.2f}s in  {cond}") 

            # Encontrar los índices (canales) donde p[0] es menor que 0.05
        significative_channels = np.where(p < threshold)[0]

        # Contar cuántos canales cumplen la condición
        num_significative_channels = len(significative_channels)
        
        # Append the significative channels to the list
        num_significative_channels_all.append(num_significative_channels)
        significative_channels_all.append(significative_channels)
        # Imprimir los canales significativos
        # print(f"Significative channels {cond} (p < 0.05):", significative_channels)
        print(f"Total number of significative channels in general {cond}:", num_significative_channels)


        # Aplicar la corrección de Benjamini-Hochberg (FDR)
        _, p_adjusted, _, _ = multipletests(p, method='fdr_bh')
        significative_channels_adjusted = np.where(p_adjusted < threshold)[0]

        # Mostrar cuántos canales siguen siendo significativos
        num_significative_channels_adjusted = len(significative_channels_adjusted)
        print(f"Significative channels {cond} after FDR-BH: {significative_channels_adjusted} de {len(channels_mag)}")
        print(f"Number of Significative channels {cond} dafter FDR-BH: {num_significative_channels_adjusted} de {len(channels_mag)}")

        
        significative_channels_adjusted_all.append(significative_channels_adjusted)
        num_significative_channels_adjusted_all.append(num_significative_channels_adjusted)
    
    
    dict_isc_window_all["significative_channels_all"]=significative_channels_all
    dict_isc_window_all["num_significative_channels_all"]=num_significative_channels_all
    dict_isc_window_all["significative_channels_adjusted_all"]=significative_channels_adjusted_all
    dict_isc_window_all["num_significative_channels_adjusted_all"]=num_significative_channels_adjusted_all
    

    # Guardar dict_isc_window_all en archivo pickle
    pickle_path = ISC_path / f"dict_isc_window_all_{cond}_{layer_script}.pkl"
    with open(pickle_path, 'wb') as f:
        pickle.dump(dict_isc_window_all, f)
        
    del array

    



🔄 Procesando condición: zinnen
Reading g:\MOUS_204\MOUS_visual\output_preproc\preproc_event\evoked_event\sub-V1001_evoked_zinnen_event-ave.fif ...
    Found the data of interest:
        t =       0.00 ...    8000.00 ms (begin_zinnen)
        0 CTF compensation matrices available
        nave = 94 - aspect type = 100
No projector specified for this dataset. Please consider the method self.add_proj.
No baseline correction applied
Reading g:\MOUS_204\MOUS_visual\output_preproc\preproc_event\evoked_event\sub-V1002_evoked_zinnen_event-ave.fif ...
    Found the data of interest:
        t =       0.00 ...    8000.00 ms (begin_zinnen)
        0 CTF compensation matrices available
        nave = 98 - aspect type = 100
No projector specified for this dataset. Please consider the method self.add_proj.
No baseline correction applied
Archivo no encontrado para sub-V1003: g:\MOUS_204\MOUS_visual\output_preproc\preproc_event\evoked_event\sub-V1003_evoked_zinnen_event-ave.fif
Archivo no encontrado 

In [7]:
dict_isc_window_all

{'iscs_window_all': [array([[-0.09948362, -0.20942709,  0.0189871 , ..., -0.26105348,
           0.00447648, -0.05778544],
         [ 0.22419642,  0.33078289,  0.42265545, ..., -0.05207814,
          -0.024384  , -0.19479147],
         [-0.10441361, -0.07340163,  0.05678539, ..., -0.16205699,
           0.24277584, -0.03125271],
         ...,
         [ 0.54662912,  0.69257853,  0.76928612, ...,  0.00663587,
           0.07963082,  0.01241193],
         [ 0.25833196,  0.4123928 ,  0.53537274, ...,  0.1105711 ,
          -0.01521799,  0.08997142],
         [ 0.34864138,  0.3325094 ,  0.27452628, ..., -0.12964312,
           0.12908107, -0.26028261]], shape=(18, 273)),
  array([[-0.28646471, -0.28412541,  0.01576695, ..., -0.1253037 ,
          -0.06119286, -0.28405957],
         [ 0.2117612 ,  0.36809213,  0.46746221, ..., -0.0833219 ,
          -0.1241995 , -0.2585841 ],
         [-0.00944529, -0.03543052,  0.0854051 , ..., -0.06155288,
           0.12876571, -0.08767179],
         ...